# Notebook 03
* Author: Melissa Ormeño; Alexis Salas
* Laboratory: Cancer Molecular Dynamics
* Date: May 2024

Aims:
* Define las mutaciones del ICGC y las clasifica por su frecuencia de alelo mutante

In [1]:
# Dependences
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import pwlf
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from scipy.stats import f_oneway, kruskal, shapiro

# Step 1: filtrar data frame 

In [2]:
# Datos con la lista de genes quinasa a trabajar 
#interpro_code = 'IPR000333'
family_name = 'STKR'
df = pd.read_csv('../data/Cancer/df_icgc_{0}.csv'.format(family_name))
print(df.shape)
print(df.columns)
df.dropna(subset='aa_mutation',inplace=True)

(62773, 44)
Index(['Gen_names', 'Uniprot Name', 'Aliases', 'Uniprot ID', 'ENS ID', 'Group',
       'icgc_mutation_id', 'icgc_donor_id', 'project_code', 'icgc_specimen_id',
       'icgc_sample_id', 'matched_icgc_sample_id', 'submitted_sample_id',
       'submitted_matched_sample_id', 'chromosome', 'chromosome_start',
       'chromosome_end', 'chromosome_strand', 'assembly_version',
       'mutation_type', 'reference_genome_allele', 'mutated_from_allele',
       'mutated_to_allele', 'total_read_count', 'mutant_allele_read_count',
       'verification_status', 'verification_platform',
       'biological_validation_status', 'biological_validation_platform',
       'consequence_type', 'aa_mutation', 'cds_mutation', 'gene_affected',
       'transcript_affected', 'gene_build_version', 'platform',
       'experimental_protocol', 'sequencing_strategy',
       'base_calling_algorithm', 'alignment_algorithm',
       'variation_calling_algorithm', 'other_analysis_algorithm',
       'seq_coverage',

/var/folders/17/bt4jbx0920sgdvwfqnkx2bnr0000gn/T/ipykernel_25283/1696366917.py:4: DtypeWarning: Columns (26,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/Cancer/df_icgc_{0}.csv'.format(family_name))


In [3]:
#Lee el contenido de algunas columnas del data frame df, que contiene el archivo "simple_somatic_mutation.open.tsv.gz" del ICGC
print(f"Assembly_version: {df['assembly_version'].unique()}")
print(f"Gene_build_version: {df['gene_build_version'].unique()}")
#print(f"base_calling_algorithm: {df['base_calling_algorithm'].unique()}")
#print(f"alignment_algorithm: {df['alignment_algorithm'].unique()}")
#print(f"experimental_protocol: {df['experimental_protocol'].unique()}")
print(f"biological_validation_status: {df['biological_validation_status'].unique()}")
print(f"biological_validation_platform: {df['biological_validation_platform'].unique()}")
print(f"verification_status: {df['verification_status'].unique()}")
print(f"mutation_type: {df['mutation_type'].unique()}")
print(f"consequence_type: {df['consequence_type'].unique()}")
print(f"mutation_type: {df['mutation_type'].unique()}")
print(f"sequencing_strategy: {df['sequencing_strategy'].unique()}")
print(f"base_calling_algorithm: {df['base_calling_algorithm'].unique()}")

Assembly_version: ['GRCh37']
Gene_build_version: [75.]
biological_validation_status: ['not tested' nan 'tested and valid']
biological_validation_platform: [nan 'PCR' 'capillary sequencing' '454 sequencing']
verification_status: ['not tested' 'tested and verified' 'tested and verified to be false']
mutation_type: ['single base substitution'
 'multiple base substitution (>=2bp and <=200bp)' 'deletion of <=200bp'
 'insertion of <=200bp']
consequence_type: ['synonymous_variant' 'missense_variant' 'stop_gained'
 'frameshift_variant' 'disruptive_inframe_insertion' 'inframe_deletion'
 'start_lost' 'stop_lost' 'stop_retained_variant']
mutation_type: ['single base substitution'
 'multiple base substitution (>=2bp and <=200bp)' 'deletion of <=200bp'
 'insertion of <=200bp']
sequencing_strategy: ['WXS' 'WGS']
base_calling_algorithm: [nan
 'CASAVA http://support.illumina.com/sequencing/sequencing_software/casava.ilmn'
 'RTA http://support.illumina.com/sequencing/sequencing_software/real-time_analy

In [4]:
# Crea una nueva columna combinando 'tipo_cancer' y 'project_code'
df['cancer_name'] = df['tipo_cancer'] + '_' + df['project_code']
# Mostrar el DataFrame para verificar
print(df['cancer_name'].unique())
df.head(2)



['Piel_SKCM-US' 'Piel_SKCA-BR' 'Piel_MELA-AU' 'Higado_LIRI-JP'
 'Higado_LINC-JP' 'Higado_LIHC-US' 'Higado_LIAD-FR' 'Higado_LICA-FR'
 'Sangre_MALY-DE' 'Sangre_LAML-US' 'Sangre_CLLE-ES' 'Sangre_LAML-KR'
 'Cerebro_GBM-US' 'Cerebro_LGG-US' 'Cerebro_PBCA-DE' 'Cerebro_PEME-CA'
 'Seno _BRCA-US' 'Seno _BRCA-EU' 'Seno _BRCA-UK' 'Seno _BRCA-FR'
 'Ovario_OV-US' 'Ovario_OV-AU' 'Pancreas_PACA-CA' 'Pancreas_PAEN-AU'
 'Pancreas_PAAD-US' 'Pancreas_PACA-AU' 'Pancreas_PAEN-IT'
 'Prostata_PRAD-US' 'Prostata_PRAD-CA' 'Prostata_PRAD-CN'
 'Prostata_PRAD-FR' 'Prostata_PRAD-UK' 'Prostata_EOPC-DE' 'Utero_UCEC-US'
 'Utero_UTCA-FR' 'Estomago_GACA-CN' 'Estomago_STAD-US' 'Pulmon_LUAD-US'
 'Pulmon_LUSC-US' 'Pulmon_LUSC-KR' 'Riñon_KIRC-US' 'Riñon_RECA-EU'
 'Riñon_KIRP-US' 'Riñon_WT-US' 'Riñon_KICH-US' 'Cabeza_HNSC-US'
 'Cabeza_THCA-US' 'Cabeza_THCA-SA' 'Cabeza_ORCA-IN' 'Vesicula_BTCA-JP'
 'Vesicula_BTCA-SG' 'Colon_READ-US' 'Colon_COAD-US' 'Cervix_CESC-US'
 'Hueso_BOCA-UK' 'Vejiga_BLCA-US']


,Gen_names,Uniprot Name,Aliases,Uniprot ID,ENS ID,Group,icgc_mutation_id,icgc_donor_id,project_code,icgc_specimen_id,...,platform,experimental_protocol,sequencing_strategy,base_calling_algorithm,alignment_algorithm,variation_calling_algorithm,other_analysis_algorithm,seq_coverage,tipo_cancer,cancer_name
1,BMPR1B,BMR1B_HUMAN,"ALK-6, ALK6, AMD3, AMDD, BDA1D, BDA2, CDw293",O00238,ENSG00000138696,TKL,MU4433033,DO37607,SKCM-US,SP82796,...,Illumina GA sequencing,NaN,WXS,NaN,NaN,TCGA-MC3 https://gdc.cancer.gov/about-data/pub...,NaN,NaN,Piel,Piel_SKCM-US
2,BMPR1B,BMR1B_HUMAN,"ALK-6, ALK6, AMD3, AMDD, BDA1D, BDA2, CDw293",O00238,ENSG00000138696,TKL,MU4433033,DO37607,SKCM-US,SP82796,...,Illumina GA sequencing,NaN,WXS,NaN,NaN,TCGA-MC3 https://gdc.cancer.gov/about-data/pub...,NaN,NaN,Piel,Piel_SKCM-US


In [5]:
#Se obtiene una lista del tipo de cancer y el proyecto paricular para cada tipo de cancer

# Obtener la lista de tipos de cáncer
list_tipo_cancer = df['tipo_cancer'].unique()

# Crear un diccionario para almacenar los resultados
resultados = {}

# Iterar sobre cada tipo de cáncer y obtener los códigos únicos de proyectos
for tipo in list_tipo_cancer:
    proyectos_unicos = df[df['tipo_cancer'] == tipo]['project_code'].unique()
    resultados[tipo] = proyectos_unicos

# Mostrar los resultados
for tipo, proyectos in resultados.items():
    print(f"{tipo}: {proyectos}")


Piel: ['SKCM-US' 'SKCA-BR' 'MELA-AU']
Higado: ['LIRI-JP' 'LINC-JP' 'LIHC-US' 'LIAD-FR' 'LICA-FR']
Sangre: ['MALY-DE' 'LAML-US' 'CLLE-ES' 'LAML-KR']
Cerebro: ['GBM-US' 'LGG-US' 'PBCA-DE' 'PEME-CA']
Seno : ['BRCA-US' 'BRCA-EU' 'BRCA-UK' 'BRCA-FR']
Ovario: ['OV-US' 'OV-AU']
Pancreas: ['PACA-CA' 'PAEN-AU' 'PAAD-US' 'PACA-AU' 'PAEN-IT']
Prostata: ['PRAD-US' 'PRAD-CA' 'PRAD-CN' 'PRAD-FR' 'PRAD-UK' 'EOPC-DE']
Utero: ['UCEC-US' 'UTCA-FR']
Estomago: ['GACA-CN' 'STAD-US']
Pulmon: ['LUAD-US' 'LUSC-US' 'LUSC-KR']
Riñon: ['KIRC-US' 'RECA-EU' 'KIRP-US' 'WT-US' 'KICH-US']
Cabeza: ['HNSC-US' 'THCA-US' 'THCA-SA' 'ORCA-IN']
Vesicula: ['BTCA-JP' 'BTCA-SG']
Colon: ['READ-US' 'COAD-US']
Cervix: ['CESC-US']
Hueso: ['BOCA-UK']
Vejiga: ['BLCA-US']


In [6]:
df['cancer_type'] = df['project_code'].apply(lambda x: x.split('-')[0])
df.head(2)

,Gen_names,Uniprot Name,Aliases,Uniprot ID,ENS ID,Group,icgc_mutation_id,icgc_donor_id,project_code,icgc_specimen_id,...,experimental_protocol,sequencing_strategy,base_calling_algorithm,alignment_algorithm,variation_calling_algorithm,other_analysis_algorithm,seq_coverage,tipo_cancer,cancer_name,cancer_type
1,BMPR1B,BMR1B_HUMAN,"ALK-6, ALK6, AMD3, AMDD, BDA1D, BDA2, CDw293",O00238,ENSG00000138696,TKL,MU4433033,DO37607,SKCM-US,SP82796,...,NaN,WXS,NaN,NaN,TCGA-MC3 https://gdc.cancer.gov/about-data/pub...,NaN,NaN,Piel,Piel_SKCM-US,SKCM
2,BMPR1B,BMR1B_HUMAN,"ALK-6, ALK6, AMD3, AMDD, BDA1D, BDA2, CDw293",O00238,ENSG00000138696,TKL,MU4433033,DO37607,SKCM-US,SP82796,...,NaN,WXS,NaN,NaN,TCGA-MC3 https://gdc.cancer.gov/about-data/pub...,NaN,NaN,Piel,Piel_SKCM-US,SKCM


In [7]:
# Crea una nueva columna combinando 'tipo_cancer' y 'project_code'
df['cancer_name'] = df['tipo_cancer'] + '_' + df['cancer_type']
# Mostrar el DataFrame para verificar
print(df['cancer_name'].unique())
df.head(2)


['Piel_SKCM' 'Piel_SKCA' 'Piel_MELA' 'Higado_LIRI' 'Higado_LINC'
 'Higado_LIHC' 'Higado_LIAD' 'Higado_LICA' 'Sangre_MALY' 'Sangre_LAML'
 'Sangre_CLLE' 'Cerebro_GBM' 'Cerebro_LGG' 'Cerebro_PBCA' 'Cerebro_PEME'
 'Seno _BRCA' 'Ovario_OV' 'Pancreas_PACA' 'Pancreas_PAEN' 'Pancreas_PAAD'
 'Prostata_PRAD' 'Prostata_EOPC' 'Utero_UCEC' 'Utero_UTCA' 'Estomago_GACA'
 'Estomago_STAD' 'Pulmon_LUAD' 'Pulmon_LUSC' 'Riñon_KIRC' 'Riñon_RECA'
 'Riñon_KIRP' 'Riñon_WT' 'Riñon_KICH' 'Cabeza_HNSC' 'Cabeza_THCA'
 'Cabeza_ORCA' 'Vesicula_BTCA' 'Colon_READ' 'Colon_COAD' 'Cervix_CESC'
 'Hueso_BOCA' 'Vejiga_BLCA']


,Gen_names,Uniprot Name,Aliases,Uniprot ID,ENS ID,Group,icgc_mutation_id,icgc_donor_id,project_code,icgc_specimen_id,...,experimental_protocol,sequencing_strategy,base_calling_algorithm,alignment_algorithm,variation_calling_algorithm,other_analysis_algorithm,seq_coverage,tipo_cancer,cancer_name,cancer_type
1,BMPR1B,BMR1B_HUMAN,"ALK-6, ALK6, AMD3, AMDD, BDA1D, BDA2, CDw293",O00238,ENSG00000138696,TKL,MU4433033,DO37607,SKCM-US,SP82796,...,NaN,WXS,NaN,NaN,TCGA-MC3 https://gdc.cancer.gov/about-data/pub...,NaN,NaN,Piel,Piel_SKCM,SKCM
2,BMPR1B,BMR1B_HUMAN,"ALK-6, ALK6, AMD3, AMDD, BDA1D, BDA2, CDw293",O00238,ENSG00000138696,TKL,MU4433033,DO37607,SKCM-US,SP82796,...,NaN,WXS,NaN,NaN,TCGA-MC3 https://gdc.cancer.gov/about-data/pub...,NaN,NaN,Piel,Piel_SKCM,SKCM


In [8]:
df['verification_status'].unique()

array(['not tested', 'tested and verified',
       'tested and verified to be false'], dtype=object)

In [9]:
#Selecciona las columnas que se utilizaran para el analisis
df_cancer= df[['Gen_names', 'icgc_donor_id', 'project_code','cancer_type', 'cancer_name','consequence_type','icgc_mutation_id', 'total_read_count', 
               'mutant_allele_read_count',
       'verification_status', 'aa_mutation', 'cds_mutation', 'tipo_cancer']]
print(df_cancer.shape)
df_cancer.head()

(7678, 13)


,Gen_names,icgc_donor_id,project_code,cancer_type,cancer_name,consequence_type,icgc_mutation_id,total_read_count,mutant_allele_read_count,verification_status,aa_mutation,cds_mutation,tipo_cancer
1,BMPR1B,DO37607,SKCM-US,SKCM,Piel_SKCM,synonymous_variant,MU4433033,21.0,8.0,not tested,L192L,576C>T,Piel
2,BMPR1B,DO37607,SKCM-US,SKCM,Piel_SKCM,synonymous_variant,MU4433033,21.0,8.0,not tested,L222L,666C>T,Piel
3,BMPR1B,DO37607,SKCM-US,SKCM,Piel_SKCM,synonymous_variant,MU4433033,21.0,8.0,not tested,L192L,576C>T,Piel
4,BMPR1B,DO37607,SKCM-US,SKCM,Piel_SKCM,synonymous_variant,MU4433033,21.0,8.0,not tested,L192L,576C>T,Piel
5,BMPR1B,DO37607,SKCM-US,SKCM,Piel_SKCM,synonymous_variant,MU4433033,21.0,8.0,not tested,L192L,576C>T,Piel


In [10]:
df_cancer = df_cancer.drop_duplicates(subset=['Gen_names', 'icgc_donor_id', 'project_code','cancer_type', 'cancer_name','consequence_type','icgc_mutation_id', 'total_read_count', 
               'mutant_allele_read_count',
       'verification_status', 'aa_mutation', 'cds_mutation', 'tipo_cancer'])
df_cancer.shape

(4729, 13)

In [11]:
print(df_cancer['total_read_count'].min())

5.0


In [12]:
df_cancer.shape
df_cancer['verification_status'].unique()

array(['not tested', 'tested and verified',
       'tested and verified to be false'], dtype=object)

In [13]:
#Crea un data frame donde solo esten los datos que han sido verificados y que no esten testeados
#Seleccionar filas donde la columna 'verification_status' sea 'tested and verified' o 'tested and verified to be false'
df_verificado = df_cancer.loc[(df_cancer['verification_status'] == 'tested and verified') | (df_cancer['verification_status'] == 'not tested')]
print(df_verificado['verification_status'].unique())
print(df_verificado.shape)

['not tested' 'tested and verified']
(4725, 13)


In [14]:
df_verificado['consequence_type'].unique()

array(['synonymous_variant', 'missense_variant', 'stop_gained',
       'frameshift_variant', 'disruptive_inframe_insertion',
       'inframe_deletion', 'start_lost', 'stop_lost',
       'stop_retained_variant'], dtype=object)

In [15]:
# elimina las mutaciones synonymous (silenciosas) y crea un data frame: df_drop
df_consequence=df_verificado[df_verificado['consequence_type'] != 'synonymous_variant'].reset_index(drop=True)
print(df_consequence.shape)

(3730, 13)


In [16]:
print(df_consequence['total_read_count'].min())

5.0


In [18]:
df_consequence[df_consequence['aa_mutation']=='A188V']

,Gen_names,icgc_donor_id,project_code,cancer_type,cancer_name,consequence_type,icgc_mutation_id,total_read_count,mutant_allele_read_count,verification_status,aa_mutation,cds_mutation,tipo_cancer
1316,ACVR1C,DO46657,PAAD-US,PAAD,Pancreas_PAAD,missense_variant,MU1887805,652.0,129.0,not tested,A188V,563C>T,Pancreas
2080,ACVR1C,DO41199,UCEC-US,UCEC,Utero_UCEC,missense_variant,MU1887805,261.0,86.0,not tested,A188V,563C>T,Utero
2084,ACVR1C,DO41199,UCEC-US,UCEC,Utero_UCEC,missense_variant,MU1887805,57.0,14.0,not tested,A188V,563C>T,Utero


In [19]:
df_consequence = df_consequence.dropna(subset='aa_mutation', axis=0)
df_consequence = df_consequence.dropna(subset='Gen_names', axis=0)
df_consequence = df_consequence.dropna(subset='total_read_count', axis=0)
df_consequence = df_consequence.dropna(subset='mutant_allele_read_count', axis=0)
df_consequence.shape

(3700, 13)

In [20]:
df_consequence_cleaned = df_consequence.dropna(subset=['Gen_names', 'icgc_donor_id', 'aa_mutation', 'consequence_type', 'tipo_cancer'], axis=0)
df_consequence_cleaned.shape

(3700, 13)

In [21]:
df_consequence_cleaned.columns

Index(['Gen_names', 'icgc_donor_id', 'project_code', 'cancer_type',
       'cancer_name', 'consequence_type', 'icgc_mutation_id',
       'total_read_count', 'mutant_allele_read_count', 'verification_status',
       'aa_mutation', 'cds_mutation', 'tipo_cancer'],
      dtype='object')

In [22]:
df_consequence_cleaned = df_consequence_cleaned.drop_duplicates(subset=['Gen_names', 'icgc_donor_id', 'project_code', 'cancer_type',
       'cancer_name', 'consequence_type', 'icgc_mutation_id',
       'total_read_count', 'mutant_allele_read_count', 'verification_status',
       'aa_mutation', 'cds_mutation', 'tipo_cancer'])
df_consequence_cleaned.shape

(3700, 13)

In [23]:
# Agrupa el df por las columnas asociadas a la mutacion, como: gen, cromosoma, mutationID, tipo de consecuencia, tipo de cancer, etc
# Con esta columnas agrupadas se suma el total_read_count, y mutant_allele_read_count y se obtiene una columna de frecuencia de alelo mutante
df_drop=df_consequence_cleaned.groupby(['Gen_names','icgc_donor_id','aa_mutation','consequence_type','tipo_cancer']).sum(numeric_only=True).reset_index()
print(df_drop.shape)
print(df_drop['total_read_count'].min())
print(df_drop[df_drop['aa_mutation']=='E63D'])
df_drop

(3321, 7)
5.0
    Gen_names icgc_donor_id aa_mutation  consequence_type tipo_cancer  \
156    ACVR1B      DO218828        E63D  missense_variant       Seno    

     total_read_count  mutant_allele_read_count  
156              61.0                      28.0  


,Gen_names,icgc_donor_id,aa_mutation,consequence_type,tipo_cancer,total_read_count,mutant_allele_read_count
0,ACVR1,DO10132,C81Y,missense_variant,Colon,33.0,6.0
1,ACVR1,DO10880,D403N,missense_variant,Cerebro,98.0,49.0
2,ACVR1,DO10880,V214A,missense_variant,Cerebro,101.0,52.0
3,ACVR1,DO13522,L337S,missense_variant,Cerebro,67.0,50.0
4,ACVR1,DO14208,W412L,missense_variant,Cabeza,34.0,3.0
...,...,...,...,...,...,...,...
3316,TGFBR2,DO8490,R403H,missense_variant,Colon,180.0,7.0
3317,TGFBR2,DO8490,R428H,missense_variant,Colon,180.0,7.0
3318,TGFBR2,DO8594,R497*,stop_gained,Colon,110.0,15.0
3319,TGFBR2,DO8594,R522*,stop_gained,Colon,110.0,15.0


In [24]:


# Supongamos que el DataFrame se llama df
# Elimina las filas donde total_read_count == 0.0
df_drop = df_drop[df_drop['total_read_count'] != 0.0]

# También puedes usar .query (opcional)
# df_filtered = df.query('total_read_count != 0.0')

# Verificar el resultado
df_drop


,Gen_names,icgc_donor_id,aa_mutation,consequence_type,tipo_cancer,total_read_count,mutant_allele_read_count
0,ACVR1,DO10132,C81Y,missense_variant,Colon,33.0,6.0
1,ACVR1,DO10880,D403N,missense_variant,Cerebro,98.0,49.0
2,ACVR1,DO10880,V214A,missense_variant,Cerebro,101.0,52.0
3,ACVR1,DO13522,L337S,missense_variant,Cerebro,67.0,50.0
4,ACVR1,DO14208,W412L,missense_variant,Cabeza,34.0,3.0
...,...,...,...,...,...,...,...
3316,TGFBR2,DO8490,R403H,missense_variant,Colon,180.0,7.0
3317,TGFBR2,DO8490,R428H,missense_variant,Colon,180.0,7.0
3318,TGFBR2,DO8594,R497*,stop_gained,Colon,110.0,15.0
3319,TGFBR2,DO8594,R522*,stop_gained,Colon,110.0,15.0


In [25]:
df_drop[df_drop['icgc_donor_id']=='DO8408']

,Gen_names,icgc_donor_id,aa_mutation,consequence_type,tipo_cancer,total_read_count,mutant_allele_read_count
537,ACVR1B,DO8408,G167E,missense_variant,Colon,444.0,213.0
538,ACVR1B,DO8408,G219E,missense_variant,Colon,444.0,213.0
3312,TGFBR2,DO8408,S441F,missense_variant,Colon,465.0,90.0
3313,TGFBR2,DO8408,S466F,missense_variant,Colon,465.0,90.0


In [26]:
df_drop[df_drop['aa_mutation']=='A188V']

,Gen_names,icgc_donor_id,aa_mutation,consequence_type,tipo_cancer,total_read_count,mutant_allele_read_count
749,ACVR1C,DO41199,A188V,missense_variant,Utero,318.0,100.0
848,ACVR1C,DO46657,A188V,missense_variant,Pancreas,652.0,129.0


In [27]:
df_drop['consequence_type'].unique()

array(['missense_variant', 'start_lost', 'stop_gained',
       'frameshift_variant', 'inframe_deletion', 'stop_lost',
       'disruptive_inframe_insertion', 'stop_retained_variant'],
      dtype=object)

In [28]:
df_drop.shape

(3321, 7)

In [29]:
df_drop['aa_mutation'].nunique()

2744

In [35]:
# Paso 1: Contar el número de mutaciones únicas por gen, donante y tipo de cáncer
mutation_count = (
    df_drop.groupby(['Gen_names', 'icgc_donor_id', 'tipo_cancer'])['aa_mutation']
    .nunique()
    .reset_index(name='mutation_count')  # Crear columna 'mutation_count' con el número de mutaciones únicas
)

# Paso 2: Sumar 'total_read_count' agrupando por gen, donante y tipo de cáncer
total_read_count = (
    df_drop.groupby(['Gen_names', 'icgc_donor_id', 'tipo_cancer'])['total_read_count']
    .sum()
    .reset_index(name='total_read_count')  # Crear columna 'total_read_count' con la suma
)

# Paso 3: Combinar los resultados de 'mutation_count' y 'read_count' en un solo DataFrame
read_count = pd.merge(mutation_count, total_read_count, on=['Gen_names', 'icgc_donor_id', 'tipo_cancer'])

# Paso 4: Ordenar por el número de mutaciones
read_count = read_count.sort_values(by='mutation_count', ascending=False)

# Mostrar el resultado final
read_count


,Gen_names,icgc_donor_id,tipo_cancer,mutation_count,total_read_count
1311,TGFBR1,DO9876,Colon,17,3464.0
1303,TGFBR1,DO8264,Colon,14,2186.0
313,ACVR1C,DO41235,Utero,14,2736.0
1245,TGFBR1,DO41235,Utero,11,2234.0
153,ACVR1B,DO27989,Ovario,10,1481.0
...,...,...,...,...,...
834,AMHR2,DO8280,Colon,1,77.0
835,AMHR2,DO8642,Colon,1,88.0
836,AMHR2,DO9010,Colon,1,43.0
838,AMHR2,DO9876,Colon,1,81.0


In [38]:
print(f"Numero total de pacientes: {read_count['icgc_donor_id'].nunique()}")
print(f"Numero total de lecturas de todos los pacientes (por millon): {read_count['total_read_count'].sum(numeric_only=True)/10**6}")

Numero total de pacientes: 1061
Numero total de lecturas de todos los pacientes (por millon): 0.390428


In [39]:
#interpro_code = 'IPR000333'
family_name = 'STKR'
ruta = '../data/mutaciones/df_count_mut_{0}.csv'.format(family_name)
df_drop.to_csv(ruta, index=False)

In [40]:
#interpro_code = 'IPR000333'
family_name = 'STKR'
ruta = '../data/mutaciones/df_read_count_mut_{0}.csv'.format(family_name)
read_count.to_csv(ruta, index=False)

# Archivo total genes

In [73]:
import pandas as pd

family_name = 'genes_normalizado'

# Definir las columnas necesarias
columnas_necesarias = ['icgc_donor_id', 'gene_affected','total_read_count', 'mutant_allele_read_count', 'aa_mutation', 'consequence_type', 'gene_name', 'tipo_cancer','verification_status']

# Lista para almacenar los datos limpios
df_cleaned = []

# Leer el archivo en fragmentos (chunks)
chunksize = 100000  # Número de filas por fragmento (ajustar según la memoria disponible)
for chunk in pd.read_csv('../data/Cancer/df_icgc_{0}.csv'.format(family_name), 
                         usecols=columnas_necesarias, 
                         chunksize=chunksize, 
                         low_memory=False):
    # Eliminar duplicados en el fragmento
    chunk = chunk.drop_duplicates(subset=['icgc_donor_id', 'gene_affected','total_read_count', 'mutant_allele_read_count', 'aa_mutation', 'consequence_type', 'gene_name', 'tipo_cancer','verification_status'])
    df_cleaned.append(chunk)

# Concatenar todos los fragmentos limpios en un solo DataFrame
df_genes_totales = pd.concat(df_cleaned, ignore_index=True)

# Mostrar las primeras filas del DataFrame limpio
print(df_genes_totales.head())


  icgc_donor_id  total_read_count  mutant_allele_read_count  \
0       DO37659             115.0                      65.0   
1       DO37659             115.0                      65.0   
2       DO37659             115.0                      65.0   
3       DO37659             115.0                      65.0   
4       DO37659             115.0                      65.0   

  verification_status         consequence_type aa_mutation    gene_affected  \
0          not tested       synonymous_variant        L59L  ENSG00000106336   
1          not tested  downstream_gene_variant         NaN  ENSG00000224729   
2          not tested           intron_variant         NaN  ENSG00000106336   
3          not tested       synonymous_variant        L77L  ENSG00000106336   
4          not tested             exon_variant         NaN  ENSG00000106336   

  tipo_cancer   gene_name  
0        Piel      FBXO24  
1        Piel  PCOLCE-AS1  
2        Piel      FBXO24  
3        Piel      FBXO24  
4     

In [74]:
df_genes_totales['verification_status'].unique()

array(['not tested', 'tested and verified',
       'tested and verified to be false', 'tested and inconclusive'],
      dtype=object)

In [75]:
# Cambiar el nombre de la columna 'gene_name' a 'Gen_names'
df_genes_totales = df_genes_totales.rename(columns={'gene_name': 'Gen_names'})


In [76]:
#Selecciona las columnas que se utilizaran para el analisis
df_cancer_total = df_genes_totales[['icgc_donor_id', 'gene_affected', 'total_read_count','aa_mutation', 'consequence_type', 'Gen_names', 'tipo_cancer','verification_status']]
print(df_cancer_total.shape)
print(df_cancer_total['verification_status'].unique())
df_cancer_total.head()

(56474376, 8)
['not tested' 'tested and verified' 'tested and verified to be false'
 'tested and inconclusive']


,icgc_donor_id,gene_affected,total_read_count,aa_mutation,consequence_type,Gen_names,tipo_cancer,verification_status
0,DO37659,ENSG00000106336,115.0,L59L,synonymous_variant,FBXO24,Piel,not tested
1,DO37659,ENSG00000224729,115.0,NaN,downstream_gene_variant,PCOLCE-AS1,Piel,not tested
2,DO37659,ENSG00000106336,115.0,NaN,intron_variant,FBXO24,Piel,not tested
3,DO37659,ENSG00000106336,115.0,L77L,synonymous_variant,FBXO24,Piel,not tested
4,DO37659,ENSG00000106336,115.0,NaN,exon_variant,FBXO24,Piel,not tested


In [77]:
#Crea un data frame donde solo esten los datos que han sido verificados y que no esten testeados
#Seleccionar filas donde la columna 'verification_status' sea 'tested and verified' o 'tested and verified to be false'
df_verificado2 = df_cancer_total.loc[(df_cancer_total['verification_status'] == 'tested and verified') | (df_cancer_total['verification_status'] == 'not tested')]
print(df_verificado2['verification_status'].unique())
print(df_verificado2.shape)

['not tested' 'tested and verified']
(56471348, 8)


In [78]:
df_verificado2['consequence_type'].unique()

array(['synonymous_variant', 'downstream_gene_variant', 'intron_variant',
       'exon_variant', 'upstream_gene_variant', 'missense_variant',
       '3_prime_UTR_variant', 'stop_gained', 'splice_region_variant',
       '5_prime_UTR_variant', 'start_lost', 'splice_acceptor_variant',
       'splice_donor_variant', 'intragenic_variant',
       'stop_retained_variant', 'frameshift_variant', 'stop_lost',
       '5_prime_UTR_premature_start_codon_gain_variant',
       'intergenic_region', 'disruptive_inframe_insertion',
       'inframe_insertion', 'inframe_deletion',
       'disruptive_inframe_deletion', nan, 'initiator_codon_variant',
       'exon_loss_variant'], dtype=object)

In [79]:
# elimina las mutaciones synonymous (silenciosas) y crea un data frame: df_drop
df_consequence_total = df_verificado2[df_verificado2['consequence_type'] != 'synonymous_variant'].reset_index(drop=True)
print(df_consequence_total.shape)

(54691807, 8)


In [81]:
df_consequence_total = df_consequence_total.dropna(subset='aa_mutation', axis=0)
df_consequence_total = df_consequence_total.dropna(subset='Gen_names', axis=0)
df_consequence_total = df_consequence_total.dropna(subset='total_read_count', axis=0)
df_consequence_total.shape

(4810812, 8)

In [82]:
print(df_consequence_total.isnull().sum())

icgc_donor_id          0
gene_affected          0
total_read_count       0
aa_mutation            0
consequence_type       0
Gen_names              0
tipo_cancer            0
verification_status    0
dtype: int64


In [83]:
df_consequence_total

,icgc_donor_id,gene_affected,total_read_count,aa_mutation,consequence_type,Gen_names,tipo_cancer,verification_status
16,DO37659,ENSG00000006607,65.0,P481S,missense_variant,FARP2,Piel,not tested
18,DO37659,ENSG00000006607,105.0,P481S,missense_variant,FARP2,Piel,not tested
21,DO37659,ENSG00000135925,11.0,P8S,missense_variant,WNT10A,Piel,not tested
23,DO37659,ENSG00000135925,64.0,P8S,missense_variant,WNT10A,Piel,not tested
24,DO37659,ENSG00000041515,36.0,P338S,missense_variant,MYO16,Piel,not tested
...,...,...,...,...,...,...,...,...
54691793,DO708,ENSG00000144285,35.0,L989I,missense_variant,SCN1A,Vejiga,not tested
54691798,DO708,ENSG00000104177,42.0,G442W,missense_variant,MYEF2,Vejiga,not tested
54691800,DO708,ENSG00000198721,97.0,W301*,stop_gained,ECI2,Vejiga,not tested
54691802,DO708,ENSG00000198721,97.0,W331*,stop_gained,ECI2,Vejiga,not tested


In [121]:
# Paso 1: Contar el número de mutaciones únicas por gen, donante y tipo de cáncer
read_count_total = (
    df_consequence_total.groupby(['Gen_names', 'icgc_donor_id', 'tipo_cancer']
                                ).agg(
        mutation_count=('aa_mutation', 'nunique'),  # Contar mutaciones únicas
        total_read_count=('total_read_count', 'sum')  # Sumar total_read_count
    )
    .reset_index()
)

# Paso 4: Ordenar por el número de mutaciones
read_count_total = read_count_total.sort_values(by='mutation_count', ascending=False)

# Mostrar el resultado final
read_count_total

,Gen_names,icgc_donor_id,tipo_cancer,mutation_count,total_read_count
1799443,TTN,DO37406,Piel,464,57471.0
1799781,TTN,DO41235,Utero,410,62471.0
1799770,TTN,DO41020,Utero,326,20127.0
1799910,TTN,DO43306,Utero,291,22766.0
1801199,TTN,DO8264,Colon,280,38897.0
...,...,...,...,...,...
1232647,PCDHGA12,DO37823,Piel,1,113.0
1232648,PCDHGA12,DO37951,Piel,1,35.0
1232649,PCDHGA12,DO38192,Estomago,1,110.0
1232650,PCDHGA12,DO38201,Estomago,1,204.0


In [138]:
# Buscar la fila específica con condiciones lógicas
resultado = df_consequence_total[
    (df_consequence_total['Gen_names'] == 'TTN') &
    (df_consequence_total['icgc_donor_id'] == 'DO37406') &
    (df_consequence_total['tipo_cancer'] == 'Piel') ]

# Mostrar el resultado
print(resultado['aa_mutation'].nunique())
print(resultado['total_read_count'].sum())

464
57471.0


In [139]:
lista_tkl = read_count['Gen_names'].unique()

In [140]:
genes_tkl = read_count_total.loc[read_count_total['Gen_names'].isin(lista_tkl)]
genes_tkl.shape
genes_tkl

,Gen_names,icgc_donor_id,tipo_cancer,mutation_count,total_read_count
1699645,TGFBR1,DO9876,Colon,17,3464.0
29468,ACVR1C,DO41235,Utero,14,2736.0
1699637,TGFBR1,DO8264,Colon,14,2186.0
1699579,TGFBR1,DO41235,Utero,11,2234.0
29308,ACVR1B,DO27989,Ovario,10,1481.0
...,...,...,...,...,...
183891,BMPR1A,DO42713,Utero,1,56.0
183890,BMPR1A,DO42568,Utero,1,36.0
183889,BMPR1A,DO42560,Utero,1,64.0
183888,BMPR1A,DO42480,Utero,1,75.0


In [141]:
genes_tkl_mut = df_consequence_total.loc[df_consequence_total['Gen_names'].isin(lista_tkl)]
genes_tkl_mut.shape
genes_tkl_mut

,icgc_donor_id,gene_affected,total_read_count,aa_mutation,consequence_type,Gen_names,tipo_cancer,verification_status
7407,DO37659,ENSG00000135409,67.0,W418*,stop_gained,AMHR2,Piel,not tested
7408,DO37659,ENSG00000135409,67.0,W115*,stop_gained,AMHR2,Piel,not tested
7412,DO37659,ENSG00000135409,78.0,W418*,stop_gained,AMHR2,Piel,not tested
7413,DO37659,ENSG00000135409,78.0,W115*,stop_gained,AMHR2,Piel,not tested
42169,DO37643,ENSG00000107779,32.0,G246*,stop_gained,BMPR1A,Piel,not tested
...,...,...,...,...,...,...,...,...
54656733,DO633,ENSG00000139567,31.0,G148D,missense_variant,ACVRL1,Vejiga,not tested
54661073,DO622,ENSG00000204217,39.0,S547F,missense_variant,BMPR2,Vejiga,not tested
54671338,DO785,ENSG00000163513,146.0,A261R?,frameshift_variant,TGFBR2,Vejiga,not tested
54671339,DO785,ENSG00000163513,146.0,A286R?,frameshift_variant,TGFBR2,Vejiga,not tested


In [142]:
genes_tkl_mut[genes_tkl_mut['aa_mutation']=='C61Y']

,icgc_donor_id,gene_affected,total_read_count,aa_mutation,consequence_type,Gen_names,tipo_cancer,verification_status
23901419,DO44888,ENSG00000107779,93.0,C61Y,missense_variant,BMPR1A,Higado,not tested
23901422,DO44888,ENSG00000107779,238.0,C61Y,missense_variant,BMPR1A,Higado,not tested


In [143]:
genes_tkl_mut.columns

Index(['icgc_donor_id', 'gene_affected', 'total_read_count', 'aa_mutation',
       'consequence_type', 'Gen_names', 'tipo_cancer', 'verification_status'],
      dtype='object')

In [144]:
genes_tkl.sort_values(by='mutation_count')

,Gen_names,icgc_donor_id,tipo_cancer,mutation_count,total_read_count
183887,BMPR1A,DO42105,Utero,1,41.0
29599,ACVR2A,DO38543,Estomago,1,153.0
29590,ACVR2A,DO37835,Piel,1,83.0
29582,ACVR2A,DO36256,Prostata,1,39.0
29573,ACVR2A,DO32972,Pancreas,1,138.0
...,...,...,...,...,...
29308,ACVR1B,DO27989,Ovario,10,1481.0
1699579,TGFBR1,DO41235,Utero,11,2234.0
1699637,TGFBR1,DO8264,Colon,14,2186.0
29468,ACVR1C,DO41235,Utero,14,2736.0


In [145]:
print(genes_tkl.isnull().sum())

Gen_names           0
icgc_donor_id       0
tipo_cancer         0
mutation_count      0
total_read_count    0
dtype: int64


In [146]:
#Datos del DF completo de genes
print(f"Numero total de pacientes: {read_count_total['icgc_donor_id'].nunique()}")
print(f"Numero total de lecturas de todos los pacientes (por millon): {read_count_total['total_read_count'].sum(numeric_only=True)/10**6}")
print(f"Numero total de mutaciones: {read_count_total['mutation_count'].sum()}")

Numero total de pacientes: 12814
Numero total de lecturas de todos los pacientes (por millon): 482.771367
Numero total de mutaciones: 4374842


In [147]:
#Datos del DF completo de genes de solo los tkl
print(f"Numero total de pacientes: {genes_tkl['icgc_donor_id'].nunique()}")
print(f"Numero total de lecturas de todos los pacientes (por millon): {genes_tkl['total_read_count'].sum(numeric_only=True)/10**6}")
print(f"Numero total de mutaciones: {genes_tkl['mutation_count'].sum()}")

Numero total de pacientes: 1061
Numero total de lecturas de todos los pacientes (por millon): 0.390428
Numero total de mutaciones: 3321


In [148]:
# Datos del data frame donde solo estan los tkl
print(f"Numero total de pacientes: {read_count['icgc_donor_id'].nunique()}")
print(f"Numero total de lecturas de todos los pacientes (por millon): {read_count['total_read_count'].sum(numeric_only=True)/10**6}")
print(f"Numero total de mutaciones: {read_count['mutation_count'].sum()}")

Numero total de pacientes: 1061
Numero total de lecturas de todos los pacientes (por millon): 0.390428
Numero total de mutaciones: 3321


In [149]:
#comparar las diferencias entre 2 data frames
import pandas as pd

# Supongamos que tienes los DataFrames df1 y df2
# df1 y df2 tienen la columna 'icgc_donor_id'

# 1. Encuentra las filas que están en df1 pero no en df2
diff_df1 = genes_tkl_mut[~genes_tkl_mut['total_read_count'].isin(df_drop['total_read_count'])]

# 2. Encuentra las filas que están en df2 pero no en df1
diff_df2 = df_drop[~df_drop['total_read_count'].isin(genes_tkl_mut['total_read_count'])]

# 3. Combina las diferencias
differences = pd.concat([diff_df1, diff_df2], ignore_index=True)

# Mostrar las diferencias
differences


,icgc_donor_id,gene_affected,total_read_count,aa_mutation,consequence_type,Gen_names,tipo_cancer,verification_status,mutant_allele_read_count
0,DO44888,ENSG00000107779,238.0,C61Y,missense_variant,BMPR1A,Higado,not tested,NaN
1,DO4036,ENSG00000138696,256.0,P65S,missense_variant,BMPR1B,Seno,not tested,NaN
2,DO4036,ENSG00000138696,256.0,P95S,missense_variant,BMPR1B,Seno,not tested,NaN
3,DO35442,ENSG00000138696,208.0,R149W,missense_variant,BMPR1B,Pancreas,not tested,NaN
4,DO35442,ENSG00000138696,208.0,R179W,missense_variant,BMPR1B,Pancreas,not tested,NaN
...,...,...,...,...,...,...,...,...,...
57,DO42673,NaN,345.0,R86H,missense_variant,TGFBR1,Utero,NaN,25.0
58,DO14338,NaN,154.0,R460C,missense_variant,TGFBR2,Cabeza,NaN,15.0
59,DO14338,NaN,154.0,R485C,missense_variant,TGFBR2,Cabeza,NaN,15.0
60,DO2323,NaN,176.0,W521*,stop_gained,TGFBR2,Seno,NaN,114.0
